In [11]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# === FOLDER SETUP ===
input_dir = 'SAVED/FOR_LOAD'
output_dir = 'SAVED/FOR_SAVE'
os.makedirs(input_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

# === PLOT STYLE ===
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 11,
    'figure.figsize': (8, 5),
    'axes.spines.top': False,
    'axes.spines.right': False
})

# === UTILITY ===
def save_plot(fig, filename):
    path = os.path.join(output_dir, filename)
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"✅ Zapisano wykres: {path}")


# === PLOTTING FUNCTIONS ===
def plot_line(df, stem, show=False):
    fig, ax = plt.subplots()
    ax.plot(df["PredictionDate"], df["ActualReturn"], label="Rzeczywiste", color="#007BFF", linewidth=2)
    ax.plot(df["PredictionDate"], df["PredictedReturn"], label="Prognozowane", color="#D14036", linestyle="--", linewidth=2)
    ax.set_title("Prognoza vs Rzeczywistość")
    ax.set_xlabel("Data")
    ax.set_ylabel("Zwrot")
    ax.legend()
    save_plot(fig, f"{stem}_1_line.png")
    if show:
        plt.show()


def plot_scatter(df, stem):
    fig, ax = plt.subplots()
    ax.scatter(df["ActualReturn"], df["PredictedReturn"], alpha=0.6, color="#007BFF", label="Obserwacje")
    min_val = min(df["ActualReturn"].min(), df["PredictedReturn"].min())
    max_val = max(df["ActualReturn"].max(), df["PredictedReturn"].max())
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', lw=1, label="Idealna prognoza")
    ax.set_title("Rozrzut: Rzeczywiste vs Prognozowane")
    ax.set_xlabel("Rzeczywiste Zwroty")
    ax.set_ylabel("Prognozowane Zwroty")
    ax.legend()
    save_plot(fig, f"{stem}_2_scatter.png")

def plot_qq(df, stem):
    residuals = df["ActualReturn"] - df["PredictedReturn"]
    fig = plt.figure()
    stats.probplot(residuals, dist="norm", plot=plt)
    plt.title("QQ-Wykres Reszt")
    plt.xlabel("Kwantyle Teoretyczne")
    plt.ylabel("Kwantyle Empiryczne")
    save_plot(fig, f"{stem}_3_qq.png")

def plot_histogram(df, stem):
    residuals = df["ActualReturn"] - df["PredictedReturn"]
    fig, ax = plt.subplots()
    sns.histplot(residuals, kde=True, ax=ax, color="#D14036", bins=30)
    ax.set_title("Histogram Reszt")
    ax.set_xlabel("Reszty (Błąd)")
    ax.set_ylabel("Częstość")
    save_plot(fig, f"{stem}_4_residual_histogram.png")

# === MAIN LOOP (with Ticker support) ===
files = [f for f in os.listdir(input_dir) if f.endswith(('.csv', '.xls', '.xlsx'))]

if not files:
    print("Brak plików .csv lub .xlsx w folderze SAVED/FOR_LOAD.")
else:
    print(f"Znaleziono {len(files)} plików. Generowanie wykresów...")

    for filename in files:
        filepath = os.path.join(input_dir, filename)
        print(f"\nPrzetwarzanie: {filename}")
        
        try:
            df = pd.read_csv(filepath) if filename.endswith(".csv") else pd.read_excel(filepath)
        except Exception as e:
            print(f"Błąd przy wczytywaniu {filename}: {e}")
            continue

        # Check required columns
        required_cols = {"PredictedReturn", "ActualReturn", "PredictionDate", "Ticker"}
        if not required_cols.issubset(df.columns):
            print(f"Plik {filename} nie zawiera wymaganych kolumn: {required_cols}")
            continue

        df["PredictionDate"] = pd.to_datetime(df["PredictionDate"])
        stem = os.path.splitext(filename)[0]

        # Group by ticker
        for ticker, group in df.groupby("Ticker"):
            if group.empty:
                continue

            print(f"  ➤ Ticker: {ticker}, {len(group)} obserwacji")

            plot_line(group, f"{stem}_{ticker}_1_line")
            plot_scatter(group, f"{stem}_{ticker}_2_scatter")
            plot_qq(group, f"{stem}_{ticker}_3_qq")
            plot_histogram(group, f"{stem}_{ticker}_4_hist")

    print("\n✅ Wszystkie wykresy zapisane w: SAVED/FOR_SAVE")



Znaleziono 1 plików. Generowanie wykresów...

Przetwarzanie: merged_eval_run01_AAAAAAAAAAAAAAAAAA_TopKPredicted_ReturnsRelativeStrengthStrategy_k5_20250626-201038.csv
  ➤ Ticker: AAPL, 31 obserwacji
✅ Zapisano wykres: SAVED/FOR_SAVE\merged_eval_run01_AAAAAAAAAAAAAAAAAA_TopKPredicted_ReturnsRelativeStrengthStrategy_k5_20250626-201038_AAPL_1_line_1_line.png
✅ Zapisano wykres: SAVED/FOR_SAVE\merged_eval_run01_AAAAAAAAAAAAAAAAAA_TopKPredicted_ReturnsRelativeStrengthStrategy_k5_20250626-201038_AAPL_2_scatter_2_scatter.png
✅ Zapisano wykres: SAVED/FOR_SAVE\merged_eval_run01_AAAAAAAAAAAAAAAAAA_TopKPredicted_ReturnsRelativeStrengthStrategy_k5_20250626-201038_AAPL_3_qq_3_qq.png
✅ Zapisano wykres: SAVED/FOR_SAVE\merged_eval_run01_AAAAAAAAAAAAAAAAAA_TopKPredicted_ReturnsRelativeStrengthStrategy_k5_20250626-201038_AAPL_4_hist_4_residual_histogram.png
  ➤ Ticker: AMZN, 31 obserwacji
✅ Zapisano wykres: SAVED/FOR_SAVE\merged_eval_run01_AAAAAAAAAAAAAAAAAA_TopKPredicted_ReturnsRelativeStrengthStrategy_